# ProperTIQ — Car-wash site suitability (Larimer County, CO)

A second worked vertical, same method, different rules. Car washes live or die on
**traffic exposure** and **competitor spacing**, so we score commercial parcels on
real **CDOT traffic counts (AADT)**, a **competitor gap**, and (optionally)
**neighborhood demand** from the Census ACS.

Everything runs on open data you fetch yourself — ProperTIQ owns the method, not
the data.


## Setup

In [1]:
# !pip install "propertiq[examples]"
import json
import os
import urllib.parse
import urllib.request
import warnings

import geopandas as gpd
import osmnx as ox

import propertiq as pq

warnings.simplefilter("ignore")


## 1 · Data (open sources)

| Layer | Source |
|---|---|
| Parcels + zoning | Larimer County GIS ArcGIS |
| Traffic (AADT) | Colorado DOT OTIS Traffic Explorer |
| Competitor car washes | OpenStreetMap (`amenity=car_wash`) |
| Demand (median income) | Census ACS 5-yr + TIGERweb block groups *(optional, needs a free key)* |

Small AOI near Loveland keeps every pull fast.


In [2]:
BBOX = (-105.10, 40.36, -105.02, 40.42)  # (W, S, E, N)


def _get(url: str) -> str:
    req = urllib.request.Request(url, headers={"User-Agent": "ProperTIQ-demo"})
    return urllib.request.urlopen(req, timeout=60).read().decode("utf-8", "replace")


def esri_bbox(layer_url: str, fields: str) -> gpd.GeoDataFrame:
    """Fetch a bbox-clipped Esri REST layer as a WGS84 GeoDataFrame."""
    w, s, e, n = BBOX
    q = {
        "where": "1=1", "geometry": f"{w},{s},{e},{n}",
        "geometryType": "esriGeometryEnvelope", "inSR": "4326",
        "spatialRel": "esriSpatialRelIntersects", "outFields": fields,
        "outSR": "4326", "returnGeometry": "true", "f": "geojson",
    }
    gj = json.loads(_get(layer_url + "/query?" + urllib.parse.urlencode(q)))
    return gpd.GeoDataFrame.from_features(gj["features"], crs=4326)


parcels = esri_bbox(
    "https://maps1.larimer.org/arcgis/rest/services/MapServices/Parcels/MapServer/3",
    "PARCELNUM,LOCCITY",
)
zoning = esri_bbox(
    "https://maps1.larimer.org/arcgis/rest/services/MapServices/LC_Zoning/MapServer/0",
    "ZONING_ABBRV_DISTRICT",
)
aadt = esri_bbox(
    "https://dtdapps.coloradodot.info/arcgis/rest/services/OTIS/TrafficExplorer/MapServer/0",
    "AADT,AADTYR,COUNTLOCATION",
)
car_washes = ox.features_from_bbox(BBOX, tags={"amenity": "car_wash"})[["geometry"]]
car_washes = car_washes.to_crs(4326).reset_index(drop=True)

print(
    f"{len(parcels)} parcels · {len(zoning)} zoning polys · {len(aadt)} traffic counts "
    f"(max AADT {int(aadt['AADT'].max())}) · {len(car_washes)} competitor car washes"
)


1000 parcels · 125 zoning polys · 25 traffic counts (max AADT 47000) · 7 competitor car washes


### Optional demand layer (Census ACS)

Median household income per block group is a useful demand signal. The Census API
now needs a free key — set `CENSUS_API_KEY` to enable it. Without one, the demo
runs fine and simply leaves demand out.


In [3]:
has_demand = False
demand = None
CENSUS_KEY = os.environ.get("CENSUS_API_KEY")
bg = esri_bbox(
    "https://tigerweb.geo.census.gov/arcgis/rest/services/TIGERweb/tigerWMS_ACS2022/MapServer/8",
    "GEOID",
)
if CENSUS_KEY:
    import pandas as pd

    url = (
        "https://api.census.gov/data/2022/acs/acs5?get=NAME,B19013_001E"
        f"&for=block%20group:*&in=state:08%20county:069&key={CENSUS_KEY}"
    )
    rows = json.loads(_get(url))
    df = pd.DataFrame(rows[1:], columns=rows[0])
    df["GEOID"] = df.state + df.county + df.tract + df["block group"]
    df["med_hh_income"] = pd.to_numeric(df["B19013_001E"], errors="coerce")
    demand = bg.merge(df[["GEOID", "med_hh_income"]], on="GEOID", how="left")
    has_demand = demand["med_hh_income"].notna().any()

print("demand layer:", "enabled" if has_demand else "skipped (set CENSUS_API_KEY to enable)")


demand layer: skipped (set CENSUS_API_KEY to enable)


## 2 · Prepare the parcels

Derive **acres** and **zoning** (parcels carry neither) and enrich each parcel
with the **AADT of its nearest traffic count** within ~800 m — that becomes the
traffic-exposure signal.


In [4]:
MEAS = "EPSG:2876"  # Colorado North (US-ft) — local, accurate for the AOI

parcels["acres"] = parcels.to_crs(MEAS).area / 43560.0

# Zoning by spatial join.
zj = gpd.sjoin(
    parcels.to_crs(MEAS),
    zoning.to_crs(MEAS)[["ZONING_ABBRV_DISTRICT", "geometry"]],
    predicate="intersects", how="left",
)
zj = zj[~zj.index.duplicated(keep="first")]
parcels["zone"] = zj["ZONING_ABBRV_DISTRICT"].reindex(parcels.index).values

# Traffic: nearest AADT count station within ~800 m (else NaN -> scored worst).
near = gpd.sjoin_nearest(
    parcels.to_crs(MEAS), aadt.to_crs(MEAS)[["AADT", "geometry"]],
    max_distance=800 * 3.28084, distance_col="_d",
)
near = near[~near.index.duplicated(keep="first")]
parcels["aadt"] = near["AADT"].reindex(parcels.index).values

print("commercial (CC) parcels:", int((parcels["zone"] == "CC").sum()))
print("parcels with traffic data:", int(parcels["aadt"].notna().sum()))


commercial (CC) parcels: 18
parcels with traffic data: 602


## 3 · The strategy

**Filters**: zoned **commercial (`CC`)** and at least **0.3 acres**.
**Scoring**: **traffic exposure** (AADT, 0.45) · **competitor gap** within 2 mi
(0.30) · **demand** (median income, 0.25 — when enabled). Weights auto-balance to
whatever criteria are active.


In [5]:
score = [
    pq.AttrValue("aadt", prefer="high", weight=0.45),     # traffic exposure
    pq.Gap(of=car_washes, within_mi=2, weight=0.30),      # underserved by competitors
]
if has_demand:
    score.append(pq.Index(demand, value_field="med_hh_income", prefer="high", weight=0.25))

car_wash = pq.Strategy(
    name="car_wash",
    filters=[
        pq.AttrIn("zone", ["CC"]),   # commercial
        pq.MinArea(acres=0.3),       # enough room for a wash + queue
    ],
    score=score,
)
car_wash


Strategy(filters=[AttrIn(field='zone', values=['CC']), MinArea(acres=0.3)], score=[AttrValue(field='aadt', weight=0.45, prefer='high', name=None), Gap(of=                                            geometry
0  POLYGON ((-105.10015 40.40763, -105.09983 40.4...
1  POLYGON ((-105.09636 40.39132, -105.09625 40.3...
2  POLYGON ((-105.09564 40.39131, -105.09564 40.3...
3  POLYGON ((-105.07247 40.41593, -105.07274 40.4...
4  POLYGON ((-105.02647 40.40691, -105.02606 40.4...
5  POLYGON ((-105.0736 40.42008, -105.0733 40.420...
6  POLYGON ((-105.05083 40.40864, -105.05055 40.4..., within_mi=2, weight=0.3, name=None, prefer='low')], name='car_wash')

## 4 · Run, inspect, map

In [6]:
result = car_wash.run(parcels)
print(f"{len(result.parcels)} candidate parcels")
result.top(10)[["PARCELNUM", "acres", "aadt", "score", "rank"]]


6 candidate parcels


,PARCELNUM,acres,aadt,score,rank
362,9525200017,4.381432,27000.0,100.000000,1
367,9525200014,2.843901,27000.0,100.000000,2
364,9524370001,2.934227,23000.0,60.000000,3
628,9524217017,0.720052,23000.0,46.666667,4
366,9524300014,3.558997,21000.0,40.000000,5
673,9524217013,0.312211,23000.0,20.000000,6


In [7]:
result.explain().head(10).round(1)

,rank,AttrValue,Gap,score
362,1,60.0,40.0,100.0
367,2,60.0,40.0,100.0
364,3,20.0,40.0,60.0
628,4,20.0,26.7,46.7
366,5,0.0,40.0,40.0
673,6,20.0,0.0,20.0


In [8]:
result.to_map(tooltip_fields=["aadt", "acres"])

## 5 · Export

Same as any ProperTIQ result — open-format candidates plus the strategy YAML that
the [config app](../app/README.md) reads and writes.


In [9]:
result.to_file("car_wash_candidates.geojson")
pq.dump_strategy(
    car_wash, "car_wash.yaml",
    layer_names={id(car_washes): "car_washes", **({id(demand): "demand"} if has_demand else {})},
)
print(open("car_wash.yaml").read())


name: car_wash
filters:
- attr_in:
    field: zone
    values:
    - CC
- min_area:
    acres: 0.3
score:
- attr_value:
    field: aadt
    prefer: high
    weight: 0.45
- gap:
    of: car_washes
    within_mi: 2
    weight: 0.3

